In [1]:
import time
import pandas as pd
import numpy as np

In [2]:
def run_stage2_repair(df_corrupted):
    print(f"\n==========================================")
    print(f"STAGE 2 SELF-HEALING LAYER (HoloClean Logic)")
    print(f"==========================================")
    
    df_repaired = df_corrupted.copy()
    
    # Track repair timing: T_sanitized
    t_start_repair = time.time()
    
    # 1. Relational Denial Constraint Enforcement
    # Rule 1: Reconstruct oldbalanceOrg if missing: oldbalanceOrg = newbalanceOrig + amount
    mask_old = df_repaired['oldbalanceOrg'].isna() & df_repaired['newbalanceOrig'].notna() & df_repaired['amount'].notna()
    df_repaired.loc[mask_old, 'oldbalanceOrg'] = df_repaired.loc[mask_old, 'newbalanceOrig'] + df_repaired.loc[mask_old, 'amount']
    
    # Rule 2: Reconstruct newbalanceOrig if missing: newbalanceOrig = oldbalanceOrg - amount
    mask_new = df_repaired['newbalanceOrig'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['amount'].notna()
    df_repaired.loc[mask_new, 'newbalanceOrig'] = df_repaired.loc[mask_new, 'oldbalanceOrg'] - df_repaired.loc[mask_new, 'amount']
    
    # Rule 3: Reconstruct amount if missing: amount = oldbalanceOrg - newbalanceOrig
    mask_amt = df_repaired['amount'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['newbalanceOrig'].notna()
    df_repaired.loc[mask_amt, 'amount'] = df_repaired.loc[mask_amt, 'oldbalanceOrg'] - df_repaired.loc[mask_amt, 'newbalanceOrig']
    
    # 2. Probabilistic Fallback Imputation for Remaining Unmapped Cells
    # For destination balances, use median conditional on transaction type
    for col in ['oldbalanceDest', 'newbalanceDest']:
        if df_repaired[col].isna().any():
            median_val = df_repaired[col].median()
            df_repaired[col] = df_repaired[col].fillna(median_val)
            
    # Final cleanup for un-reconstructable origin values using median
    for col in ['amount', 'oldbalanceOrg', 'newbalanceOrig']:
        if df_repaired[col].isna().any():
            df_repaired[col] = df_repaired[col].fillna(df_repaired[col].median())

    t_end_repair = time.time()
    delta_t_repair = t_end_repair - t_start_repair
    
    print(f"Self-Healing Completed in (Delta T_repair): {delta_t_repair:.4f} seconds")
    print(f"Remaining Missing Values: {df_repaired[['amount', 'oldbalanceOrg', 'newbalanceOrig']].isna().sum().sum()}")
    
    return df_repaired, delta_t_repair

# Quick Execution Test
if __name__ == "__main__":
    # Test on the 5% corrupted dataset generated earlier
    try:
        noisy_data = pd.read_csv('paysim_noise_5.csv')
        repaired_data, repair_time = run_stage2_repair(noisy_data)
    except FileNotFoundError:
        print("Please ensure 'paysim_noise_5.csv' exists before running the standalone test.")


STAGE 2 SELF-HEALING LAYER (HoloClean Logic)
Self-Healing Completed in (Delta T_repair): 1.1691 seconds
Remaining Missing Values: 0
